# Lesson 04 — Mercury's extra 43 arcseconds

**Phase 4 of the gravsim project, part I: one body curves spacetime.**

In 1859 Urbain Le Verrier, who had found Neptune from Uranus' misbehaviour, reported that
Mercury's perihelion advanced $38''$ per century faster than the planets could explain
(Newcomb's 1882 value: $43''$). A planet inside Mercury's orbit — "Vulcan" — was proposed, searched
for, and even "seen" by amateurs, but never confirmed. In November 1915 Einstein computed the
perihelion shift of the then-new field equations: $43''$. "For a few days I was beside myself
with joyous excitement," he wrote.

This lesson reproduces that number *from the simulation*, in three independent ways, and then
makes it visible.

By the end you should be able to

1. write the first post-Newtonian (1PN) correction and estimate its size from $v^2/c^2$;
2. derive the perihelion-advance formula $\Delta\varpi = 6\pi GM/(c^2 a(1-e^2))$ per orbit;
3. measure $42.98''$/cy in a two-body run, in a nine-body run, and *from the Horizons data*;
4. explain why reducing $c$ makes GR visible and what limits the trick;
5. say what the PPN parameters $\beta,\gamma$ do to the result.

## 1. The physics: the 1PN correction

In the weak-field, slow-motion limit of General Relativity the acceleration of a test body
in the field of a mass $GM$ at rest is, in harmonic coordinates,

$$
\mathbf a = -\frac{GM}{r^3}\,\mathbf r
+ \frac{GM}{c^2 r^3}\left[\left(4\frac{GM}{r} - v^2\right)\mathbf r + 4\,(\mathbf r\cdot\mathbf v)\,\mathbf v\right]
+ O(c^{-4}).
$$

The bracket is the **1PN correction**. Relative to the Newtonian term it is of order
$GM/(c^2 r) \sim v^2/c^2$, which for Mercury ($v \approx 48$ km/s on average) is $\sim2\times10^{-8}$: an animation with
GR "on" looks identical to one with it "off". Yet it is **velocity-dependent** (the
$(\mathbf r\cdot\mathbf v)\mathbf v$ term), which matters for the integrator (Lesson 01, §6), and
it is not a central $1/r^2$ force, so the eccentricity vector is no longer conserved.

**Secular effect.** Averaging the perturbation over one orbit (or solving the Binet equation
perturbatively, Lesson 05) gives the advance of the perihelion per orbit

$$
\Delta\varpi = \frac{6\pi GM}{c^2\,a\,(1-e^2)} ,
$$

which for Mercury ($a = 0.387$ AU, $e = 0.2056$) is $5.02\times10^{-7}$ rad per 88-day orbit,
i.e. $42.98''$ per Julian century. For Venus, Earth and Mars the same formula gives 8.62, 3.84
and 1.35$''$/cy: the effect falls as $1/(a(1-e^2))$ and, per century, also as $1/P$.

**PPN form.** Alternative metric theories of gravity are parametrised by $\beta$ (how
non-linear is the superposition of gravity) and $\gamma$ (how much space curvature per unit
mass). The correction becomes $\big[(2(\beta+\gamma)\,GM/r - \gamma v^2)\,\mathbf r +
2(1+\gamma)(\mathbf r\cdot\mathbf v)\mathbf v\big]$ and the advance is multiplied by
$(2+2\gamma-\beta)/3$. GR has $\beta=\gamma=1$. gravsim's `make_pn1(c, central, beta, gamma)`
implements exactly this bracket; `make_newton_pn1` adds the Newtonian term.

**Validity.** The expansion parameter is $v^2/c^2$. Reducing $c$ (a knob we will use to make
the effect visible) is legitimate as long as $v/c$ stays small; at $c/100$ Mercury has
$v/c = 0.013$ and the next order is $\sim2\times10^{-4}$ of the 1PN term.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from gravsim import constants as C
from gravsim.analysis import (
    fit_secular_rate,
    gr_precession_arcsec_per_century,
    perihelion_longitude_series,
    perihelion_passages,
    perihelion_rate_difference,
    separation,
    state_to_elements,
    wrap_angle,
)
from gravsim.bodies import Trajectory
from gravsim.data.horizons import load_j2000, load_reference_series
from gravsim.forces import accel_newton, make_newton_pn1, make_pn1
from gravsim.integrators import integrate_scipy
from gravsim.viz import BODY_COLORS, body_colors, figure_grid, plot_precession, plot_separation

plt.rcParams["figure.dpi"] = 100


def daily_grid(t0, years, step=1.0):
    return t0 + step * np.arange(int(round(years * C.JULIAN_YEAR_DAYS / step)) + 1)


def elements(state, body, central="Sun"):
    r, v = state.relative_to(central)
    i = state.index(body)
    gm = float(state.gm[state.index(central)] + state.gm[i])
    el = state_to_elements(r[i], v[i], gm)
    return {k: float(el[k]) for k in ("a", "e", "inc", "period")} | {"gm": gm}

## 2. Orders of magnitude

Before any integration: how big is the correction, and how big is the effect?

In [ ]:
two = load_j2000().subset(("Sun", "Mercury")).to_barycentric()
el = elements(two, "Mercury")
r, v = two.relative_to("Sun")
i_m = two.index("Mercury")
pn1 = make_pn1(c=C.C_AU_DAY, central=two.index("Sun"))
a_newton = accel_newton(two.pos, two.vel, two.gm, 0.0)[i_m]
a_pn1 = pn1(two.pos, two.vel, two.gm, 0.0)[i_m]
speed = np.linalg.norm(v[i_m])
print(f"Mercury at J2000: v = {speed * C.AU_KM / C.DAY_S:.1f} km/s,  v^2/c^2 = {(speed / C.C_AU_DAY) ** 2:.2e},  "
      f"GM/(c^2 r) = {C.GM_SUN / (C.C_AU_DAY ** 2 * np.linalg.norm(r[i_m])):.2e}")
print(f"|a_1PN| / |a_Newton| = {np.linalg.norm(a_pn1) / np.linalg.norm(a_newton):.2e}")
per_orbit = 6 * math.pi * el["gm"] / (C.C_AU_DAY ** 2 * el["a"] * (1 - el["e"] ** 2))
print(f"advance per orbit = {per_orbit:.3e} rad = {per_orbit * C.ARCSEC_PER_RAD:.4f} arcsec; "
      f"per century = {per_orbit * C.ARCSEC_PER_RAD * C.JULIAN_CENTURY_DAYS / el['period']:.3f} arcsec")
print("\nanalytic 1PN advance [arcsec/century] for each planet:")
full = load_j2000()
for name in C.PLANETS:
    e = elements(full, name)
    print(f"  {name:8s} {float(gr_precession_arcsec_per_century(e['a'], e['e'], e['gm'])):7.3f}")

## 3. Measurement I: Sun and Mercury

Two runs on the same daily grid with DOP853: Newton, and Newton + 1PN. The toolkit of
Lesson 03 does the rest. The dense fit of the 1PN run *alone* already gives $42.98$ (there are
no planets to wobble it), and the difference against the Newton run gives it with an error
of a few hundredths.

In [ ]:
t_grid = daily_grid(two.t, 10.0)
newton2 = integrate_scipy(two, accel_newton, t_grid)
gr2 = integrate_scipy(two, make_newton_pn1(c=C.C_AU_DAY, central=two.index("Sun")), t_grid)

t, w_n = perihelion_longitude_series(newton2, "Mercury")
_, w_g = perihelion_longitude_series(gr2, "Mercury")
fit_g = fit_secular_rate(t, w_g)
rate, err = perihelion_rate_difference(gr2, newton2, "Mercury")
analytic = float(gr_precession_arcsec_per_century(el["a"], el["e"], el["gm"]))
print(f"analytic                      {analytic:8.3f} arcsec/cy")
print(f"1PN run, dense fit            {fit_g.rate_arcsec_per_century:8.3f} ± {fit_g.rate_err_arcsec_per_century:.3f}")
print(f"(1PN - Newton), fit           {rate:8.3f} ± {err:.3f}")
print(f"reference (Einstein 1915)     {C.MERCURY_PRECESSION_GR_ARCSEC_CY:8.3f}")

fig, axes = figure_grid(2, panel=(5.6, 3.8))
plot_precession(t, w_n, ax=axes[0], label="Newton")
plot_precession(t, w_g, fit_g.rate, ax=axes[0], label="Newton + 1PN")
axes[0].set_title("Sun + Mercury: ϖ(t)")
d = wrap_angle(w_g - w_n)
plot_precession(t, d, fit_secular_rate(t, d).rate, ax=axes[1], label="(Newton + 1PN) minus Newton")
axes[1].set_title("the GR advance alone")

## 4. Measurement II: with the planets

The same experiment in the nine-body system. Now the dense series wobbles by tens of
arcseconds (Lesson 02) and its slope is $\sim570''$/cy; the 1PN run has the same wobble plus
$43''$. Only the difference shows the GR term cleanly — and it is the same $42.98$. The other
inner planets come out as well, to within the periodic terms that ten years do not average out
(EMB is the worst case; the century run of `tests/test_validation.py` gives 3.85 against 3.84).

In [ ]:
state = load_j2000()
t_grid = daily_grid(state.t, 10.0)
newton9 = integrate_scipy(state, accel_newton, t_grid)
gr9 = integrate_scipy(state, make_newton_pn1(c=C.C_AU_DAY, central=state.index("Sun")), t_grid)

t, w_n = perihelion_longitude_series(newton9, "Mercury")
_, w_g = perihelion_longitude_series(gr9, "Mercury")
fig, axes = figure_grid(2, panel=(5.6, 3.8))
plot_precession(t, w_n, fit_secular_rate(t, w_n).rate, ax=axes[0], label="Newton, Sun + 8 planets")
plot_precession(t, w_g, fit_secular_rate(t, w_g).rate, ax=axes[0], label="Newton + 1PN")
axes[0].set_title("Mercury in the nine-body run")
d = wrap_angle(w_g - w_n)
plot_precession(t, d, fit_secular_rate(t, d).rate, ax=axes[1], label="(Newton + 1PN) minus Newton")
axes[1].set_title("run-vs-run difference")

print(f"{'body':8s} {'1PN - Newton [arcsec/cy]':>26s} {'analytic':>10s}")
measured = {}
for name in ("Mercury", "Venus", "EMB", "Mars"):
    rate, err = perihelion_rate_difference(gr9, newton9, name)
    e = elements(state, name)
    measured[name] = (rate, err)
    print(f"{name:8s} {rate:16.3f} ± {err:6.3f} {float(gr_precession_arcsec_per_century(e['a'], e['e'], e['gm'])):10.3f}")

### The $1/(a(1-e^2))$ law

Converting each planet's rate back to an advance *per orbit* and plotting it against
$p = a(1-e^2)$ puts the four measured points on the analytic line
$\Delta\varpi = 6\pi GM_\odot/(c^2 p)$. The outer planets complete less than one orbit in ten
years, so only the analytic values are drawn for them.

In [ ]:
fig, axes = figure_grid(1, panel=(6.5, 4.4))
ax = axes[0]
ps, cols = [], body_colors(C.PLANETS)
for name, col in zip(C.PLANETS, cols):
    e = elements(state, name)
    p = e["a"] * (1 - e["e"] ** 2)
    ps.append(p)
    if name in measured:
        rate, err = measured[name]
        per_orbit = rate * e["period"] / C.JULIAN_CENTURY_DAYS
        ax.errorbar(p, per_orbit, yerr=err * e["period"] / C.JULIAN_CENTURY_DAYS, fmt="o", color=col, ms=6, label=f"{name} (measured)")
    else:
        ax.plot(p, 6 * math.pi * C.GM_SUN / (C.C_AU_DAY ** 2 * p) * C.ARCSEC_PER_RAD, "s", color=col, ms=5, mfc="none", label=f"{name} (analytic)")
grid = np.geomspace(min(ps) / 1.5, max(ps) * 1.5, 100)
ax.plot(grid, 6 * math.pi * C.GM_SUN / (C.C_AU_DAY ** 2 * grid) * C.ARCSEC_PER_RAD, "-", color="0.3", lw=1, label="6πGM/(c² p)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("p = a(1 - e²) [AU]"); ax.set_ylabel("perihelion advance per orbit [arcsec]")
ax.legend(fontsize=7, ncol=2); ax.grid(True, which="both", lw=0.3)
ax.set_title("The 1PN advance across the solar system")

## 5. Measurement III: Le Verrier's anomaly from the data

Horizons (DE441) contains GR. A Newtonian run drifts away from it *along Mercury's orbit*,
because its perihelion lags behind by $43''$ per century; the 1PN run does not. This is the
anomaly the way Le Verrier met it: as a residual between the best Newtonian model and the
observations, growing linearly with time. (The remaining 1PN residual is the Moon, the
asteroids and the EIH mutual terms of Phase 5.)

In [ ]:
ref = load_reference_series()
mask = ref.t <= t_grid[-1] + 1e-9
ref10 = Trajectory(ref.t[mask], ref.pos[mask], ref.vel[mask], ref.gm, ref.names)
idx = np.searchsorted(newton9.t, ref10.t)
sub = lambda tr: Trajectory(tr.t[idx], tr.pos[idx], tr.vel[idx], tr.gm, tr.names)
sep_n = separation(sub(newton9), ref10, "Mercury", t_atol=1e-9)
sep_g = separation(sub(gr9), ref10, "Mercury", t_atol=1e-9)
fig, axes = figure_grid(1, panel=(7.0, 4.0))
plot_separation(ref10.t, sep_n, ax=axes[0], label="Newton vs Horizons")
plot_separation(ref10.t, sep_g, ax=axes[0], label="Newton + 1PN vs Horizons")
axes[0].set_yscale("log")
axes[0].set_title("Mercury: distance of the runs from the ephemeris")
print(f"after 10 years: Newton {sep_n[-1] * C.AU_KM:.0f} km from Horizons, 1PN {sep_g[-1] * C.AU_KM:.0f} km")

## 6. Seeing it: the exaggeration knob

With the real $c$ the ellipse turns by $0.1''$ per orbit; a thousand years of orbits drawn on top
of each other would still look like one ellipse. The 1PN term scales as $1/c^2$, so dividing $c$
by 100 multiplies the advance by $10^4$: $0.29°$ per orbit, a visible rosette within a few
hundred orbits, while $v/c = 0.013$ keeps the expansion honest. The fitted rate must still match the
analytic formula with the same $c$ — a stringent test of both the force law and the toolkit.

In [ ]:
c_small = C.C_AU_DAY / 100.0
n_orbits = 200
t_ros = two.t + np.arange(0.0, n_orbits * el["period"], 0.25)
rosette = integrate_scipy(two, make_newton_pn1(c=c_small, central=two.index("Sun")), t_ros)
newton_ros = integrate_scipy(two, accel_newton, t_ros)
predicted = float(gr_precession_arcsec_per_century(el["a"], el["e"], el["gm"], c=c_small))
t_r, w_r = perihelion_longitude_series(rosette, "Mercury")
fit_r = fit_secular_rate(t_r, w_r)
diff_r, _ = perihelion_rate_difference(rosette, newton_ros, "Mercury")
tp, wp = perihelion_passages(rosette, "Mercury")
print(f"c/100: v/c = {speed / c_small:.3f}")
print(f"analytic:    {predicted:10.0f} arcsec/cy = {predicted / (C.JULIAN_CENTURY_DAYS / el['period']) / 3600:.3f} deg/orbit")
print(f"dense fit:   {fit_r.rate_arcsec_per_century:10.0f}   ratio {fit_r.rate_arcsec_per_century / predicted:.4f}")
print(f"differenced: {diff_r:10.0f}   ratio {diff_r / predicted:.4f}")
print(f"passages:    {fit_secular_rate(tp, wp).rate_arcsec_per_century:10.0f}   ratio {fit_secular_rate(tp, wp).rate_arcsec_per_century / predicted:.4f}")

In [ ]:
r_rel, _ = rosette.relative("Mercury", "Sun")
t_rel = rosette.t - rosette.t[0]
fig = plt.figure(figsize=(12.0, 5.0), layout="constrained")
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
shown = range(0, n_orbits, 20)
cmap = plt.get_cmap("viridis")
for k in shown:
    m = (t_rel >= k * el["period"]) & (t_rel <= (k + 1) * el["period"])
    ax3.plot(r_rel[m, 0], r_rel[m, 1], r_rel[m, 2], color=cmap(k / n_orbits), lw=0.9,
             label=f"orbit {k + 1}" if k in (0, max(shown)) else None)
ax3.plot([0], [0], [0], "k*", ms=9, label="Sun")
lim = np.abs(r_rel).max() * 1.05
ax3.set_xlim(-lim, lim); ax3.set_ylim(-lim, lim); ax3.set_zlim(-lim, lim); ax3.set_box_aspect((1, 1, 1))
ax3.set_xlabel("x [AU]"); ax3.set_ylabel("y [AU]"); ax3.set_zlabel("z [AU]")
ax3.view_init(elev=60, azim=-60); ax3.legend(loc="upper left", fontsize=8)
ax3.set_title(f"Mercury with c/100: every 20th of {n_orbits} orbits")
ax2 = fig.add_subplot(1, 2, 2)
plot_precession(t_r, w_r, fit_r.rate, ax=ax2, units="deg", label="Newton + 1PN, c/100")
ax2.set_title("the perihelion really turns")

How far can the knob go? At $c/1000$, $v/c = 0.13$ and the 2PN terms ($\sim v^4/c^4$, a few
percent) are no longer negligible: the 1PN rate will differ from the exact (Schwarzschild) one
at the percent level. Lesson 05 computes the exact orbit and shows exactly where 1PN stops
being enough.

## 7. What if gravity were a different theory? (PPN preview)

The advance scales as $(2+2\gamma-\beta)/3$. With the $c/100$ knob the runs are short, so we can
try a few $(\beta,\gamma)$ in seconds. A theory with $\gamma = 1/2$ (scalar–tensor gravity with a
weak scalar) would have given Einstein $2/3$ of the answer; $\beta = 0$ would give $4/3$ of it.
The 1976 Viking Shapiro-delay measurement and Cassini (2003) pinned $\gamma-1 = (2.1\pm2.3)\times10^{-5}$.

In [ ]:
t_ppn = two.t + np.arange(0.0, 30 * el["period"], 0.25)
print(f"{'beta':>5s} {'gamma':>6s} {'(2+2γ-β)/3':>11s} {'measured/GR':>12s}")
base = None
for beta, gamma in ((1.0, 1.0), (1.0, 0.5), (0.0, 1.0), (1.0, 0.0)):
    tr = integrate_scipy(two, make_newton_pn1(c=c_small, central=two.index("Sun"), beta=beta, gamma=gamma), t_ppn)
    tn = integrate_scipy(two, accel_newton, t_ppn)
    rate, _ = perihelion_rate_difference(tr, tn, "Mercury")
    if base is None:
        base = rate
    print(f"{beta:5.1f} {gamma:6.1f} {(2 + 2 * gamma - beta) / 3:11.3f} {rate / base:12.3f}")

## 8. Summary

* The 1PN correction is $\sim10^{-8}$ of Newton's force and velocity-dependent; its secular
  effect is $6\pi GM/(c^2 a(1-e^2))$ per orbit: $42.98''$/century for Mercury.
* Measured three ways — two-body, nine-body (by differencing), and as a residual against the
  Horizons ephemeris — the simulation gives $42.98$ to a few hundredths.
* Dividing $c$ by 100 makes the rosette visible; the measured rate still matches the formula
  to $0.2\%$. The PPN factor $(2+2\gamma-\beta)/3$ is reproduced.

## Exercises

1. Repeat §3 with `rtol=1e-10`. The dense fit of the 1PN run changes; does the *difference*?
2. At which value of $c$ does the differenced rate deviate from the formula by 1 %? (Use 30-orbit
   runs and halve $c$ each time. Compare $v/c$ at that point with Lesson 05's exact result.)
3. Replace `make_newton_pn1` by `combine(accel_newton, make_pn1(central=...))` and check that the
   runs are bit-for-bit identical.
4. ★ The 1PN run in §5 still drifts from Horizons by a few km after ten years. Which of the
   omitted ingredients (Moon, asteroids, EIH planet–planet terms, solar $J_2$) do you expect to
   dominate? Estimate each.
5. ★ Compute Venus' rate over a century (`daily_grid(state.t, 100)`, about ten seconds) and
   compare with 8.62. Why does Venus, nearly circular, need the longer span?